In [ ]:
# Notebook 03 - EXTRAÇÃO DAS FEATURES

import os
import numpy as np
import pandas as pd

# 1. Carregar o dataset intermediário limpo
caminho_interim = "../data/interim/df_interim_limpo.csv"
if not os.path.exists(caminho_interim):
    caminho_interim = "../data/interim/dataset_dados_emoçoes.csv"

print("Carregando dados intermediários...")
df_interim = pd.read_csv(caminho_interim)
print(f"Total de registros carregados: {len(df_interim)}")

# Exibir colunas para conferência
print("Colunas disponíveis no DataFrame:", df_interim.columns.tolist())

# 2. Função BCEA otimizada com fator de probabilidade
def calcular_bcea(x, y, probabilidade=0.68):
    if len(x) < 2 or len(y) < 2:
        return 0.0
    mask = (x != 0) & (y != 0)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        return 0.0
    
    sig_x = np.std(x, ddof=1)
    sig_y = np.std(y, ddof=1)
    
    correlacao = np.corrcoef(x, y)[0, 1]
    if np.isnan(correlacao):
        correlacao = 0.0
        
    k = -2 * np.log(1 - probabilidade)
    bcea = 2 * np.pi * sig_x * sig_y * np.sqrt(max(0.0, 1 - correlacao**2)) * k
    return bcea

# 3. Agrupamento por Sujeito, Janela e Target (usando as colunas reais)
resultados_features = []
grupos = df_interim.groupby(["sujeito_id", "janela_id", "target"])

print("Extraindo features espaciais (BCEA e Médias)...")
for (sujeito, janela, target_val), grupo in grupos:
    x = grupo["GazeX"].values
    y = grupo["GazeY"].values

    bcea_val = calcular_bcea(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    resultados_features.append({
        "sujeito_id": sujeito,
        "janela_id": janela,
        "target": target_val,
        "BCEA": bcea_val,
        "Mean_X": mean_x,
        "Mean_Y": mean_y,
    })

df_features = pd.DataFrame(resultados_features)
print(f"\nFeatures extraídas com sucesso! Total de janelas processadas: {len(df_features)}")
display(df_features.head())

# 4. Salvar o dataset 
os.makedirs("../data/processed", exist_ok=True)
caminho_saida_final = "../data/processed/df_mega.csv"
df_features.to_csv(caminho_saida_final, index=False)
print(f"Dataset final salvo com sucesso em '{caminho_saida_final}'.")

Carregando dados intermediários...


/tmp/ipykernel_64236/1205172560.py:13: DtypeWarning: Columns (0: Pupil Diameter Left (mm), 1: Pupil Diameter Right (mm)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_interim = pd.read_csv(caminho_interim)


Total de registros carregados: 440041
Colunas disponíveis no DataFrame: ['No.', 'Name', 'Time (ms)', 'Timegap (ms)', 'Duration', 'GazeX', 'GazeY', 'Pupil Diameter Left (mm)', 'Pupil Diameter Right (mm)', 'X-offset', 'Y-offset', 'Mouse X-coord', 'Mouse Y-coord', 'AOI', 'Left Eye X', 'Left Eye Y', 'Right Eye X', 'Right Eye Y', 'Detection', 'sujeito_id', 'target', 'janela_id', 'Eye-Tracker ID']
Extraindo features espaciais (BCEA e Médias)...

Features extraídas com sucesso! Total de janelas processadas: 457


,sujeito_id,janela_id,target,BCEA,Mean_X,Mean_Y
0,T12,0,0,51656.125613,394.114,120.143
1,T12,1,0,36014.106027,383.980,144.051
2,T12,2,0,24428.052635,378.676,161.570
3,T12,3,0,43144.059677,383.000,140.147
4,T12,4,0,43404.563192,377.530,141.507


Dataset final salvo com sucesso em '../data/processed/df_mega.csv'.


In [10]:
import os
import glob
import pandas as pd
import numpy as np

print("Iniciando a varredura e consolidação dos dados brutos...")

# 1. Garantir que a pasta de destino existe
os.makedirs('../data/processed', exist_ok=True)

# 2. Localizar os arquivos na pasta csv ou raw
caminho_busca = '../data/csv/*.csv'
arquivos = glob.glob(caminho_busca)

if not arquivos:
    caminho_busca = 'data/raw/*.csv'
    arquivos = glob.glob(caminho_busca)

print(f"Total de arquivos encontrados: {len(arquivos)}")

lista_gaze = []
lista_geral = []

def calcular_bcea(gx, gy):
    """Calcula a Bivariate Contour Ellipse Area (BCEA) para os eixos X e Y do olhar."""
    mask = (gx != 0) & (gy != 0) & (~gx.isna()) & (~gy.isna())
    gx_c, gy_c = gx[mask], gy[mask]
    if len(gx_c) < 3:
        return 0.0
    sx, sy = gx_c.std(), gy_c.std()
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_c, gy_c)[0, 1] if len(gx_c) > 1 else 0.0
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

# 3. Processamento individual de cada participante
for arq in arquivos:
    nome_arquivo = os.path.basename(arq)
    sujeito_id = ''.join(filter(str.isdigit, nome_arquivo)) or nome_arquivo
    
    df_temp = pd.read_csv(arq)
    
    # Padroniza os nomes das colunas (remove espaços e converte para minúsculas para facilitar a busca)
    df_temp.columns = [str(c).strip() for c in df_temp.columns]
    
    # Identifica dinamicamente as colunas de X e Y do olhar
    col_x = next((c for c in df_temp.columns if 'gazex' in c.lower() or c.lower() == 'x' or 'pos_x' in c.lower()), None)
    col_y = next((c for c in df_temp.columns if 'gazey' in c.lower() or c.lower() == 'y' or 'pos_y' in c.lower()), None)
    
    if col_x and col_y:
        gx = pd.to_numeric(df_temp[col_x], errors='coerce')
        gy = pd.to_numeric(df_temp[col_y], errors='coerce')
    else:
        # Fallback: pega as duas primeiras colunas numéricas
        gx = pd.to_numeric(df_temp.iloc[:, 0], errors='coerce')
        gy = pd.to_numeric(df_temp.iloc[:, 1], errors='coerce')
    
    # Identifica coluna de emoção/condição se existir
    col_emocao = next((c for c in df_temp.columns if 'emoc' in c.lower() or 'cond' in c.lower() or 'emotion' in c.lower()), None)
    
    # Cria um dataframe limpo e padronizado para este sujeito específico
    df_sujeito_limpo = pd.DataFrame({
        'subject_id': sujeito_id,
        'GazeX': gx,
        'GazeY': gy
    })
    
    if col_emocao:
        df_sujeito_limpo['Emotion'] = df_temp[col_emocao].values
        
    lista_gaze.append(df_sujeito_limpo)
    
    # Cálculo do BCEA geral para o dicionário consolidado
    bcea_geral = calcular_bcea(gx, gy)
    
    dados_sujeito = {
        'subject_id': sujeito_id,
        'BCEA_Geral': bcea_geral
    }
    
    if col_emocao:
        for emocao, grupo in df_temp.groupby(col_emocao):
            g_x_sub = pd.to_numeric(grupo[col_x], errors='coerce') if col_x else pd.to_numeric(grupo.iloc[:, 0], errors='coerce')
            g_y_sub = pd.to_numeric(grupo[col_y], errors='coerce') if col_y else pd.to_numeric(grupo.iloc[:, 1], errors='coerce')
            b_emoc = calcular_bcea(g_x_sub, g_y_sub)
            dados_sujeito[f'BCEA_{emocao}'] = b_emoc
            
    lista_geral.append(dados_sujeito)

# 4. Consolidando e Salvando os Arquivos Finais sem conflitos de estrutura
print("\nConsolidando e salvando dataframes...")
df_gaze_com_emocoes = pd.concat(lista_gaze, ignore_index=True)
df_gaze_com_emocoes.to_csv('../data/processed/gaze_com_emocoes.csv', index=False)

df_geral_consolidado = pd.DataFrame(lista_geral)
df_geral_consolidado.to_csv('../data/processed/dataset_dados_geral.csv', index=False)
df_geral_consolidado.to_csv('../data/processed/dataset_features_avancadas.csv', index=False)
df_geral_consolidado.to_csv('../data/processed/dataset_dados_emocoes.csv', index=False)

print("Processamento concluído com sucesso! Todos os 4 arquivos foram gerados na pasta data/processed/")

Iniciando a varredura e consolidação dos dados brutos...
Total de arquivos encontrados: 41

Consolidando e salvando dataframes...
Processamento concluído com sucesso! Todos os 4 arquivos foram gerados na pasta data/processed/
